# etm notebook

## vscode shortcuts

- ctrl + =: reset editor group size (make window widths equal)
- cmd + k + s: show shortcuts

- cmd + `:   show/hide terminal
- ctrl + b:  show/hide panel
- shift + cmd + e: switch view to panel and back
- shift + cmd + f: switch to find
- cmd + k, z: zen mode
- shift + cmd + b: open command palette

- cmd + p: file explorer
- cmd + up/down: beginning/end of file
- shift + F12: show uses for cursor item
- F12: goto definition of cursor item
- **F2: rename all instances of cursor variable**
- opt + up/down: move line up/down
- shift + opt + up/down: copy line up/down

- opt + cmd + left/right: previous/next tab



all relevant hashes in dataview

schedule item loop -> get_item_views(item, beg_week, end_week)

get_item_views updates the relevant dataview hashes including the weekly views for the range of weeks using either caches or creating as needed

create_item_views calls the various handlers such as handle_today

# Engage brain before starting engine!!!

## Item Class

** See class Item below for example **

### Instance Methods

- relevant() - return the relevant datetime for the item or None

- alerts() - return a list of alerts occurring today for the item or []

- agenda(beg_dt, end_dt) - return a list of agenda rows for the range of dates. Empty list if there are no instances in range of dates. When range includes the current date, include beginbys, pastdues and inbox

- effort(beg_dt, end_dt) - return a list of effort rows for the range of dates. Empty list if there are no instances in range of dates

- ditto for other weekly views

- history() - return the history row for the item

- ditto for other non-weekly views

- weekly_views(beg_wk, end_wk) - return a hash with view keys and lists of corresponding view rows for the range of weeks.

    - weekly view names: agenda, busy, effort, completed

    - data structure for weekly views: view_name -> list of (iso-datetime, view row) tuples where iso-datetime is the integer tuple (year, week number, weekday number, 24-hour hour number, minute number)


## Views

- iterate through all items appending relevant 

- agenda (cached?). Create collects agenda rows from all items, sorts and makes tree

# TODO

-  

- Clean up Dataview and Item classes - what's actually needed for schedule?

- 

## Unit Tests

In [ ]:
# my_module.py
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

In [ ]:
# pytest actions
# 1) Discover all files matching test_*.py or *_test.py.
# 2) Run all functions and methods prefixed with test_.

# illustrative pytest "test_*.py" file for my_module.py above
# tests/test_my_module.py
import pytest
from my_module import add, subtract

def test_add():
    assert add(1, 2) == 3
    assert add(-1, 1) == 0
    assert add(0, 0) == 0

def test_subtract():
    assert subtract(2, 1) == 1
    assert subtract(2, 2) == 0
    assert subtract(0, 1) == -1

if __name__ == "__main__":
    pytest.main()

In [14]:
import re
from dateutil.rrule import rruleset, rrule, DAILY
from datetime import datetime, timedelta
import pytz

class Item:
    def __init__(self, json_dict=None, input_string=None):
        self.created = self._get_current_timestamp()
        self.itemtype = None
        self.summary = None
        self.start = None
        self.end = None
        self.recurrence = None
        self.rruleset = None
        if json_dict:
            self._init_from_json(json_dict)
        elif input_string:
            self.parse_input(input_string)

    def _get_current_timestamp(self):
        return datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%S")

    def _init_from_json(self, json_dict):
        self.created = json_dict.get("created", self.created)
        self.itemtype = json_dict.get("itemtype")
        self.summary = json_dict.get("summary")
        self.start = self._parse_datetime(json_dict.get("s", "").replace("{T}:", ""))
        self.end = self._parse_datetime(json_dict.get("e", "").replace("{T}:", ""))
        self.recurrence = json_dict.get("r", [{}])[0]

    def parse_input(self, input_string):
        tokens = self._tokenize(input_string)
        self._parse_tokens(tokens)
        self._validate()

    def _tokenize(self, input_string):
        pattern = r'(@\w+ [^@&]+)|(&\w+ \S+)|(^\S+)|(\S[^@&]*)'
        matches = re.findall(pattern, input_string)
        return [match[0] or match[1] or match[2] or match[3] for match in matches if match[0] or match[1] or match[2] or match[3]]

    def _parse_tokens(self, tokens):
        self.itemtype = tokens[0][0]
        summary_tokens = []
        recurrence_attributes = {}
        for token in tokens[1:]:
            if token.startswith('@'):
                break
            summary_tokens.append(token)
        self.summary = ' '.join(summary_tokens)
        for token in tokens[len(summary_tokens) + 1:]:
            if token.startswith('@s'):
                self.start = self._parse_datetime(token[3:].strip())
            elif token.startswith('@e'):
                self.end = self._parse_duration(token[3:].strip())
            elif token.startswith('@r'):
                self.recurrence = self._parse_recurrence(token[3:].strip())
            elif token.startswith('&'):
                attribute, value = self._parse_attribute(token)
                recurrence_attributes[attribute] = value
        if self.recurrence:
            self.recurrence.update(recurrence_attributes)

    def _parse_datetime(self, datetime_str):
        if not datetime_str:
            return None
        try:
            return datetime.strptime(datetime_str, "%Y%m%dT%H%M%S")
        except ValueError:
            try:
                return datetime.strptime(datetime_str, "%Y/%m/%d")
            except ValueError:
                raise ValueError(f"Invalid datetime format: {datetime_str}")

    def _parse_duration(self, duration_str):
        match = re.match(r'(\d+)([dwmy])', duration_str)
        if not match:
            raise ValueError(f"Invalid duration format: {duration_str}")
        value, unit = match.groups()
        if unit == 'd':
            return timedelta(days=int(value))
        elif unit == 'w':
            return timedelta(weeks=int(value))
        elif unit == 'm':
            return timedelta(days=int(value) * 30)
        elif unit == 'y':
            return timedelta(days=int(value) * 365)

    def _parse_recurrence(self, recurrence_str):
        return {"r": recurrence_str}

    def _parse_attribute(self, attribute_str):
        key, value = attribute_str[1:].split()
        if key == "M":
            value = [int(value)]
        elif key == "w":
            value = [f"{value[:1]}TH"]
        return key, value

    def _validate(self):
        if self.itemtype == '*' and not self.start:
            raise ValueError("Events must have a start datetime (@s)")
        if self.recurrence and not self.start:
            raise ValueError("Items with recurrence (@r) must have a start datetime (@s)")

    def to_dict(self):
        data = {
            "created": self.created,
            "itemtype": self.itemtype,
            "summary": self.summary,
        }
        if self.start:
            data["s"] = "{T}:" + self.start.strftime("%Y%m%dT%H%M%S")
        if self.end:
            data["e"] = "{T}:" + (self.start + self.end).strftime("%Y%m%dT%H%M%S")
        if self.recurrence:
            data["r"] = [self.recurrence]
        return data

    def __repr__(self):
        return str(self.to_dict())

# Example usage
json_entry = {
    "created": "{T}:20240712T1052",
    "itemtype": "*",
    "summary": "Thanksgiving",
    "s": "{T}:20101126T0500",
    "r": [
        {
            "r": "y",
            "M": [11],
            "w": ["{W}:4TH"]
        }
    ],
    "modified": "{T}:20240712T1054"
}

item_from_json = Item(json_dict=json_entry)
print(item_from_json)

item_from_string = Item(input_string="* Thanksgiving @s 2010/11/26 @r y &M 11 &w 4TH")
print(item_from_string)

{'created': '{T}:20240712T1052', 'itemtype': '*', 'summary': 'Thanksgiving', 's': '{T}:20101126T050000', 'r': [{'r': 'y', 'M': [11], 'w': ['{W}:4TH']}]}
{'created': '20240712T111404', 'itemtype': '*', 'summary': 'Thanksgiving ', 's': '{T}:20101126T000000', 'r': [{'r': 'y', 'M': [11], 'w': ['4TH']}]}


In [11]:
import re
from dateutil.rrule import rruleset, rrule, DAILY
from datetime import datetime, timedelta
import pytz

class Item:
    def __init__(self, input_string=None):
        self.created = self._get_current_timestamp()
        self.itemtype = None
        self.summary = None
        self.start = None
        self.end = None
        self.recurrence = None
        self.rruleset = None
        if input_string:
            self.parse_input(input_string)

    def _get_current_timestamp(self):
        return datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%S")

    def parse_input(self, input_string):
        tokens = self._tokenize(input_string)
        self._parse_tokens(tokens)
        self._validate()

    def _tokenize(self, input_string):
        pattern = r'(@\w+ [^@]+)|(^\S+)|(\S[^@]*)'
        matches = re.findall(pattern, input_string)
        return [match[0] or match[1] or match[2] for match in matches if match[0] or match[1] or match[2]]

    def _parse_tokens(self, tokens):
        self.itemtype = tokens[0][0]
        summary_tokens = []
        for token in tokens[1:]:
            if token.startswith('@'):
                break
            summary_tokens.append(token)
        self.summary = ' '.join(summary_tokens)
        for token in tokens[len(summary_tokens) + 1:]:
            if token.startswith('@s'):
                self.start = self._parse_datetime(token[3:].strip())
            elif token.startswith('@e'):
                self.end = self._parse_duration(token[3:].strip())
            elif token.startswith('@r'):
                self.recurrence = self._parse_recurrence(token[3:].strip())
            # Add additional parsing as needed

    def _parse_datetime(self, datetime_str):
        try:
            return datetime.strptime(datetime_str, "%Y/%m/%d")
        except ValueError:
            raise ValueError(f"Invalid datetime format: {datetime_str}")

    def _parse_duration(self, duration_str):
        match = re.match(r'(\d+)([dwmy])', duration_str)
        if not match:
            raise ValueError(f"Invalid duration format: {duration_str}")
        value, unit = match.groups()
        if unit == 'd':
            return timedelta(days=int(value))
        elif unit == 'w':
            return timedelta(weeks=int(value))
        elif unit == 'm':
            return timedelta(days=int(value) * 30)
        elif unit == 'y':
            return timedelta(days=int(value) * 365)

    def _parse_recurrence(self, recurrence_str):
        # Implement recurrence parsing logic
        pass

    def _validate(self):
        if self.itemtype == '*' and not self.start:
            raise ValueError("Events must have a start datetime (@s)")
        if self.recurrence and not self.start:
            raise ValueError("Items with recurrence (@r) must have a start datetime (@s)")

    def to_dict(self):
        data = {
            "created": self.created,
            "itemtype": self.itemtype,
            "summary": self.summary,
        }
        if self.start:
            data["s"] = self.start.strftime("%Y%m%dT%H%M%S")
        if self.end:
            data["e"] = (self.start + self.end).strftime("%Y%m%dT%H%M%S")
        if self.recurrence:
            data["r"] = self.recurrence
        return data

    def __repr__(self):
        return str(self.to_dict())

# Example usage
item = Item("* carpe diem @s 2024/7/10 @r d")
print(item)

item2 = Item("- ask ChatGPT how to fix my code")
print(item2)

{'created': '20240712T104645', 'itemtype': '*', 'summary': 'carpe diem ', 's': '20240710T000000'}
{'created': '20240712T104645', 'itemtype': '-', 'summary': 'ask ChatGPT how to fix my code'}


In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day
    
    @classmethod
    def from_string(cls, date_str):
        year, month, day = map(int, date_str.split('-'))
        return cls(year, month, day)

date1 = Date(2024, 7, 5)
date2 = Date.from_string("2024-07-05")

In [43]:
class Calculator:
    constants = {
        "pi": 3.14,
        "e": 2.71}
    
    @classmethod
    def add(cls, a, b):
        return cls.constants.get(a, a) + cls.constants.get(b, b)
    
    @classmethod
    def subtract(cls, a, b):
        return cls.constants.get(a, a) - cls.constants.get(b, b)

print(Calculator.add(5, 3))
print(Calculator.subtract(5, 3))
print(Calculator.add(5, "pi"))

8
2
8.14


In [44]:
from datetime import datetime
from dateutil import rrule
from dateutil.rrule import rruleset, DAILY, rrulestr
from dateutil.tz import gettz
import textwrap

print(f"{dict(a='three', b='two')}")

# def is_naive(dt):
#     return dt.tzinfo is None or dt.tzinfo.utcoffset(dt) is None

# Function to create a string representation of the rruleset
def rruleset_to_string(rruleset_obj):
    parts = []
    # parts.append("rrules:")
    for rule in rruleset_obj._rrule:
        # parts.append(f"{textwrap.fill(str(rule))}")
        parts.append(f"{'\\n'.join(str(rule).split('\n'))}")
    # parts.append("exdates:")
    for exdate in rruleset_obj._exdate:
        parts.append(f"EXDATE:{exdate}")
    # parts.append("rdates:")
    for rdate in rruleset_obj._rdate:
        parts.append(f"RDATE:{rdate}")
    return "\n".join(parts)

# Define the timezone (replace 'America/New_York' with your specific timezone)
pacific = gettz('US/Pacific')
mountain = gettz('America/Denver')
central = gettz('US/Central')
eastern = gettz('America/New_York')
local = gettz()
utc = gettz('UTC')
naive = None

tz = eastern
# Define the start date
start_date = datetime(2024, 10, 28, 13, 30, tzinfo=tz)  # 0:30 on Mon Oct 28, 2024

start_date = dt_to_naive(start_date)

def dt_to_naive(dt):
    return dt if dt.tzinfo is None or dt.tzinfo.utcoffset(dt) is None else dt.replace(tzinfo=None)
    

rules_lst = []

# Create a recurrence rule for daily events
rule1 = rrule.rrule(freq=DAILY, dtstart=start_date, count=14)
rules_lst.append(str(rule1))
# Create another recurrence rule for specific days (e.g., every 2 days)
rule2 = rrule.rrule(freq=DAILY, dtstart=start_date, interval=2, count=7)
rules_lst.append(str(rule2))


{'a': 'three', 'b': 'two'}

rules:
DTSTART:20241028T133000\nRRULE:FREQ=DAILY;COUNT=14
DTSTART:20241028T133000\nRRULE:FREQ=DAILY;INTERVAL=2;COUNT=7
EXDATE:2024-11-04 13:30:00
RDATE:2024-11-04 13:45:00
RDATE:2024-11-05 15:15:00

occurrences from rules:
  Mon 2024-10-28 13:30  
  Tue 2024-10-29 13:30  
  Wed 2024-10-30 13:30  
  Thu 2024-10-31 13:30  
  Fri 2024-11-01 13:30  
  Sat 2024-11-02 13:30  
  Sun 2024-11-03 13:30  
  Mon 2024-11-04 13:45  
  Tue 2024-11-05 13:30  
  Tue 2024-11-05 15:15  
  Wed 2024-11-06 13:30  
  Thu 2024-11-07 13:30  
  Fri 2024-11-08 13:30  
  Sat 2024-11-09 13:30  
  Sun 2024-11-10 13:30  

list of string representations of rules:
DTSTART:20241028T133000
RRULE:FREQ=DAILY;COUNT=14
DTSTART:20241028T133000
RRULE:FREQ=DAILY;INTERVAL=2;COUNT=7
RDATE:20241104T134500
RDATE:20241105T151500
EXDATE:20241104T133000

from list of string representations to new rules:
DTSTART:20241028T133000\nRRULE:FREQ=DAILY;COUNT=14
DTSTART:20241028T133000\nRRULE:FREQ=DAILY;INTERVAL=2;

In [ ]:

# Create an rruleset
rules = rruleset()

# Add the rules to the rruleset
rules.rrule(rule1)
rules.rrule(rule2)

# Add a specific date to include
plusdates = [datetime(2024, 11, 4, 13, 45, tzinfo=tz), datetime(2024, 11, 5, 15, 15, tzinfo=tz)]
for dt in plusdates:
    dt = dt_to_naive(dt)
    rules.rdate(dt)  # dt_to_naive(dt)
    rules_lst.append(dt.strftime("RDATE:%Y%m%dT%H%M%S"))

# Add a specific date to exclude
minusdates = [datetime(2024, 11, 4, 13, 30, tzinfo=tz),]
for dt in minusdates:
    dt = dt_to_naive(dt)
    rules.exdate(dt)
    rules_lst.append(dt.strftime("EXDATE:%Y%m%dT%H%M%S"))

# Generate the occurrences of the event
occurrences = list(rules)

print(f"\nrules:")
print(rruleset_to_string(rules))

# Print the occurrences
print("\noccurrences from rules:")
for occurrence in occurrences:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

print("\nlist of string representations of rules:")
print('\n'.join(rules_lst))

print("\nfrom list of string representations to new rules:")
rules_from_str = rrulestr('\n'.join(rules_lst))
print(rruleset_to_string(rules_from_str))

occurrences_from_str = list(rules_from_str)
print("\noccurrences from new rules:")
for occurrence in occurrences_from_str:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

In [45]:
import re

pattern = r'\b-?(?:[1-4]?(?:MO|TU|WE|TH|FR|SA|SU))\b'
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

matches = re.findall(pattern, test_string)
print(matches)

['MO', '1TU', '4FR', 'WE', 'SA', '3MO', '2WE']


In [97]:
import re

pattern = re.compile(r'\b(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b')
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

matches = re.findall(pattern, test_string)
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [48]:
import re

# Define the regex pattern
pattern = r'\b(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [49]:
import re

# Define the regex pattern
pattern = r'\b(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [50]:
import re

# Define the regex pattern
pattern = r'\b(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [56]:
import re

# Define the regex pattern
pattern = r'\b(\-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [65]:
import re

# Define the regex pattern
pattern = r'\b(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"

# Use re.finditer to find all matches
matches = re.finditer(pattern, test_string)

# Extract and print the matches
result = [(m.group(1), m.group(2)) for m in matches]
print(result)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [63]:
import re

# Define the regex pattern
pattern = r'\b((-[1-4]|[1-4])?(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
# test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, 5FR, XYZ, -5MO"
test_string = "-1TU"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('1', 'TU')]


In [99]:
import re

# Define the regex pattern
pattern = re.compile(r'\b(-[1-4]|[1-4]?)(MO|TU|WE|TH|FR|SA|SU)\b')
# pattern = re.compile(r'\b([1-4])?(MO|TU|WE|TH|FR|SA|SU)\b')

# Example test string
test_string = "MO, -1TU, 4FR, 1WE, SA, -3MO, +2WE, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('1', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE')]


In [104]:
import re

# Define the regex pattern
pattern = r'\b(-[1-4]|[1-4])(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('1', 'TU'), ('4', 'FR'), ('3', 'MO'), ('2', 'WE'), ('4', 'FR')]


In [106]:
import re

# Define the regex pattern
pattern = r'\b(-[1-4]|[1-4])?(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE'), ('4', 'FR')]


In [108]:
import re

# Define the regex pattern
pattern = r'\b(-?[1-4])?(MO|TU|WE|TH|FR|SA|SU)\b'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Print the matches
print(matches)

[('', 'MO'), ('1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('3', 'MO'), ('2', 'WE'), ('4', 'FR')]


In [116]:
import re

# Define the regex pattern
pattern = re.compile(r'(?<![\w-])(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)(?!\w)')

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

def maybe_int(x):
    try:
        return int(x)
    except ValueError:
        return x
    
good = [f"{maybe_int(x[0])}{x[1]}" for x in matches]
all = [x.strip() for x in test_string.split(',')]
bad = [x for x in all if x not in good]

print("good:", good)
print("all:", all)
print("bad:", bad)

# Print the matches
print(matches)

good: ['MO', '-1TU', '4FR', 'WE', 'SA', '-3MO', '2WE', '-4FR']
all: ['MO', '-1TU', '4FR', 'WE', 'SA', '-3MO', '2WE', '-4FR', '5FR', 'XYZ', '-5MO']
bad: ['5FR', 'XYZ', '-5MO']
[('', 'MO'), ('-1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('-3', 'MO'), ('2', 'WE'), ('-4', 'FR')]


In [112]:
import re

# Define the regex pattern
pattern = r'(?<![\w-])(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)(?!\w)'

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

# Find all non-matching parts
non_matches = re.split(pattern, test_string)
print("non matches:", non_matches)

# Filter out empty strings from non-matches
non_matches = [part for part in non_matches if part and not re.fullmatch(pattern, part)]

# Print the matches and non-matches
print("Matches:", matches)
print("Non-matches:", non_matches)

non matches: ['', '', 'MO', ', ', '-1', 'TU', ', ', '4', 'FR', ', ', '', 'WE', ', ', '', 'SA', ', ', '-3', 'MO', ', ', '2', 'WE', ', ', '-4', 'FR', ', 5FR, XYZ, -5MO']
Matches: [('', 'MO'), ('-1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('-3', 'MO'), ('2', 'WE'), ('-4', 'FR')]
Non-matches: [', ', '-1', ', ', '4', ', ', ', ', ', ', '-3', ', ', '2', ', ', '-4', ', 5FR, XYZ, -5MO']
